###  LLM에게 도구를 붙인다
- LLM 은 기본적으로 질문만 하면 답변하는 모델
- 우리가 만든 함수를 골라 사용하도록 한다
- AI 에이전트의 핵심 - RAG도 마찬가지

In [ ]:
- LLM 에게 이렇게 이야기 하는 방식
- 너에게 이런저런 도구가 있다
- LLM이 필요할 때 (내가 만일 날씨 질문을 하려면 , 날씨 조회 도구 필요)
- 알아서 그 도구 선택 후 실행 -> AI 에이전트
- LLM 이 스스로 판단해서 도구를 쓰고 결과도 확인해야 최조결과를 주는 방식
#LLM이 함수를 사용하는것이 아니라 결과를 받아 사용자에게 응답하는 구조
- 작동방식
    - 함수실행은 우리가 하고, 그 결과를 다시 LLM이 받아 자연스러운 구조로 느껴지게 함

### 전체 흐름 
- LLM에게 사용 가능한 tool 목록 알려줌
- LLM이 필요한 함수를 실행하라고 요청
- 우리가 그 함수 실행
- 함수 실행 결과를 LLM에게 돌려주면 LLM이 최종 답을 만들어 반환

- LLM -> 판단
- 우리 --> 실행

In [ ]:
## 셀 1. 라이브러리와 클라이언트 준비

import json   # JSON 처리
import os # 파일 존재 여부와 경로 처리
from datetime import datetime # 저장 시간 기록
import pandas as pd # 판다스
from dotenv import load_dotenv # 환경변수 불러오기
from openai import OpenAI # OpenAI 클라이언트

load_dotenv() # .env 파일에서 OPENAI_API_KEY 불러오기
client = OpenAI() # OpenAI 클라이언트 생성

In [2]:
df = pd.read_csv("../data/11-1_뉴스정제.csv")
df.head()

,제목,본문,카테고리,요약,출처URL,정제본문
0,현대백화점그룹 더현대 광주 추진,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...,경제,"6 6일 현대백화점그룹이 광주시에 문화복합몰을 만든다고 6일 밝혔으며, 광주시는 서...",https://n.news.naver.com/mnews/article/001/001...,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...
1,이스타항공 이상직 회사와 무관…오해 살 언동 말아야,전주 뉴시스 김얼 기자 이스타항공 자금 배임·횡령으로 전주교도소에 수감됐었던 이상직...,경제,이이스항공은 자금 배임·횡령으로 전주교도소에 수감됐었던 이상직 전 의원이 출소한 것...,https://n.news.naver.com/mnews/article/003/001...,전주 뉴시스 김얼 기자 이스타항공 자금 배임 횡령으로 전주교도소에 수감됐었던 이상직...
2,농협은행 농협금융 출범 10주년 기념주화 NFT 이벤트,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 ‘10주년 기념주...,경제,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 소셜미디어 인스타...,https://n.news.naver.com/mnews/article/366/000...,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 10주년 기념주화...
3,오늘부터 유류세 인하 폭 확대…하반기 바뀌는 세제·금융 정책은,img tag s 지난 30일 서울의 한 주유소. 〈사진 연합뉴스〉 img tag ...,경제,정부는 고유가 상황에 따라 국민의 유류비 부담 완화를 위해 이날부터 유류세를 법정 ...,https://n.news.naver.com/mnews/article/437/000...,img tag s 지난 30일 서울의 한 주유소 사진 연합뉴스 img tag e 오...
4,푸르덴셜생명 더 큰 드림 변액연금보험Ⅱ에 신규펀드 13종 추가,파이낸셜뉴스 푸르덴셜생명보험은 급변하는 금융시장에 대응하기 위해 무배당 더 큰 드림...,경제,지난르덴셜생명보험은 급변하는 금융시장에 대응하기 위해 무배당 더 큰 드림 변액연금보...,https://n.news.naver.com/mnews/article/014/000...,파이낸셜뉴스 푸르덴셜생명보험은 급변하는 금융시장에 대응하기 위해 무배당 더 큰 드림...


In [20]:
# 임시로 함수를 만들어 사용
# 날씨 조회 -> 사실 웹검색
def get_weather (city):
        #실제로는 날씨 웹검색이 들어가야하는 거지만 일단 임시 고정값
        return f"{city}의 날씨는 맑음 기온 40도"
print(get_weather("신대방"))

신대방의 날씨는 맑음 기온 40도


#### 도구 LLM 설명
- 이름 설명 인자
- 툴에 그자세한 명세 작성
- 함수 이름은 영문 설명은 LLM 이해할 수 있게 작성

In [3]:
tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "특정 도시의 현재 날씨를 조회한다",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "날씨를 알고 싶은 도시 이름"},
            },
            "required": ["city"],
        },
    },
}]
print("도구 정의 완료")

도구 정의 완료


### 2. LLM 이 도구 호출 결정권 부여


In [6]:
messages = [

    {"role":"user", "content":"신대방 날씨 어떠해?"}
    
]
response = client.chat.completions.create(

    model="gpt-5.6-luna",
    tools = tools,
    reasoning_effort="none",  #함수 도구 호출 시 필수
    messages=messages
)

In [7]:
response.choices[0].message.tool_calls

[ChatCompletionMessageFunctionToolCall(id='call_So3xrymDaQTzzlHOHZPMiteA', function=Function(arguments='{"city":"신대방"}', name='get_weather'), type='function')]

In [11]:
call = response.choices[0].message.tool_calls[0]
call

ChatCompletionMessageFunctionToolCall(id='call_So3xrymDaQTzzlHOHZPMiteA', function=Function(arguments='{"city":"신대방"}', name='get_weather'), type='function')

In [13]:
print(call.function.name)

get_weather


In [14]:
print(call.function.arguments)

{"city":"신대방"}


In [15]:
response.choices[0].message.content

3. 우리가 실제 사용 함수

In [16]:
args = call.function.arguments
type(args)

str

In [18]:
import json
args = json.loads(call.function.arguments) # 딕셔너리로 형변환

In [21]:
tool_result = get_weather(**args)
print(tool_result)

신대방의 날씨는 맑음 기온 40도


In [ ]:
### 4. 결과를 LLM에게 돌려서 최종 답을 만들기

In [22]:
messages.append(response.choices[0].message)
messages

[{'role': 'user', 'content': '신대방 날씨 어떠해?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_So3xrymDaQTzzlHOHZPMiteA', function=Function(arguments='{"city":"신대방"}', name='get_weather'), type='function')])]

In [23]:
messages.append({"role":"tool", "tool_call_id":call.id, "content":tool_result})

#3. tool 사용한 결과
messages

[{'role': 'user', 'content': '신대방 날씨 어떠해?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_So3xrymDaQTzzlHOHZPMiteA', function=Function(arguments='{"city":"신대방"}', name='get_weather'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'call_So3xrymDaQTzzlHOHZPMiteA',
  'content': '신대방의 날씨는 맑음 기온 40도'}]

In [24]:
response1 = client.chat.completions.create(

    model="gpt-5.6-luna",
    tools = tools,
    reasoning_effort="none",  #함수 도구 호출 시 필수
    messages=messages
)

In [25]:
print(response1.choices[0].message.content)

신대방은 현재 **맑고, 기온은 40도**예요. 너무 더우니 수분 섭취와 외출 시 주의하세요.


In [ ]:
# 하나의 함수로 만들어 사용

In [ ]:
available_tools = {"get_weather": get_weather}

def chat_with_tools(question, tools):
    messages = [{"role": "user", "content": question}]
    r = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
                                       reasoning_effort="none", messages=messages)
    calls = r.choices[0].message.tool_calls
    if not calls:                       # 도구가 필요 없으면 바로 답
        return r.choices[0].message.content
    messages.append(r.choices[0].message)
    for tc in calls:                    # 필요한 도구를 모두 실행
        args = json.loads(tc.function.arguments)
        result = available_tools[tc.function.name](**args)
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    r2 = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
                                        reasoning_effort="none", messages=messages)
    return r2.choices[0].message.content

print(chat_with_tools("부산 날씨 알려줘", tools))

### Agent = LLM + 도구
- 날씨 조회 도구
    - 비서 에이전트 만들고 싶다면 -> 필요한 기능?
    - 필요한 일정 정보 등록 도구
    - 일정정보 조회 도구
    - 일정 삭제 도구
-일기장을 쓰는 기능
    - 파일에 내요을 쓰는 도구
    - 그림추가 도구


In [28]:
## 다른 툴 추가
def recommend_cloth(data):
    return f"{data}의 스타일은 너무 멋져요 80년대 스타일 같아요"

tools = [{
    "type": "function",
    "function": {
        "name": "recommend",
        "description": "옷에 대해 물어보면 코멘트를 해준다",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "코멘트 받고 싶은 옷 이름"},
            },
            "required": ["city"],
        },
    },
},
{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "특정 도시의 현재 날씨를 조회한다",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "날씨를 알고 싶은 도시 이름"},
            },
            "required": ["city"],
        },
    },
}

]

In [29]:
available_tools = {"get_weather": get_weather, "recommend_cloth":recommend_cloth}

def chat_with_tools(question, tools):
    messages = [{"role": "user", "content": question}]
    r = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
                                       reasoning_effort="none", messages=messages)
    calls = r.choices[0].message.tool_calls
    if not calls:                       # 도구가 필요 없으면 바로 답
        return r.choices[0].message.content
    messages.append(r.choices[0].message)
    for tc in calls:                    # 필요한 도구를 모두 실행
        args = json.loads(tc.function.arguments)
        result = available_tools[tc.function.name](**args)
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    r2 = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
                                        reasoning_effort="none", messages=messages)
    return r2.choices[0].message.content

print(chat_with_tools("부산 날씨 알려줘", tools))

현재 부산은 **맑고 기온은 40°C**입니다. 매우 더우니 수분을 충분히 섭취하고 외출 시 햇볕을 피하세요.


### 데이터 프레임에서 뉴스 내용 조회 후 요약해주는 도구

In [38]:
# 다른 툴 추가하기 - csv 에서 제목을 물어보거나 본문을 물어볼 수 있다. -> 
def check_data(column):
    df = pd.read_csv("../data/11-1_뉴스정제.csv")

    return df[column][0]    # 제목에 대해서 물어보면 제목을, 본문을 물어보면 본문을 조회하는 도구
tools = [{
    "type": "function",
    "function": {
        "name": "recommend_cloth",
        "description": "옷에 대해서 물어봤을때 코멘트를 해준다. ",
        "parameters": {
            "type": "object",
            "properties": {
                "data": {"type": "string", "description": "코멘트 받고 싶은 옷 이름"},
            },
            "required": ["data"],
        },
    },
},
{
    "type": "function",
    "function": {
        "name": "check_data",
        "description": "데이터프레임에서 내용 조회할때 칼럼명으로 조회해서 알려준다. ",
        "parameters": {
            "type": "object",
            "properties": {
                "column": {"type": "string", "description": "조회하고 싶은 칼럼"},
            },
            "required": ["column"],
        },
    },
},
{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "특정 도시의 현재 날씨를 조회한다",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "날씨를 알고 싶은 도시 이름"},
            },
            "required": ["city"],
        },
    },
}

]

In [39]:
available_tools = {"get_weather": get_weather, 
                   "recommend_cloth" : recommend_cloth,
                   "check_data": check_data
                   }

def chat_with_tools(question, tools):
    messages = [{"role": "user", "content": question}]
    r = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
                                       reasoning_effort="none", messages=messages)
    calls = r.choices[0].message.tool_calls
    if not calls:                       # 도구가 필요 없으면 바로 답
        return r.choices[0].message.content
    messages.append(r.choices[0].message)
    for tc in calls:                    # 필요한 도구를 모두 실행
        args = json.loads(tc.function.arguments)
        result = available_tools[tc.function.name](**args)
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    r2 = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
                                        reasoning_effort="none", messages=messages)
    return r2.choices[0].message.content

print(chat_with_tools("부산 날씨 알려줘", tools))

현재 부산은 **맑고, 기온은 40°C**입니다. ☀️


In [40]:
print(chat_with_tools("이 청바지 어때?", tools))

청바지 스타일 너무 멋져요! 80년대 레트로 감성이 느껴져서 개성 있고, 캐주얼하게 입기 좋겠어요.


In [41]:
print(chat_with_tools("데이터프레임에서 본문 알려줘", tools))

서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울과 같은 문화복합몰을 만든다고 6일 밝혔다.


In [45]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "recommend_cloth",
            "description": "옷에 대해 물어보면 코멘트를 제공한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "data": {
                        "type": "string",
                        "description": "코멘트를 받고 싶은 옷 이름"
                    }
                },
                "required": ["data"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_data",
            "description": "데이터프레임에서 입력한 컬럼의 내용을 조회한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "column": {
                        "type": "string",
                        "description": "조회하고 싶은 컬럼명"
                    }
                },
                "required": ["column"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "특정 도시의 현재 날씨를 조회한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "날씨를 알고 싶은 도시 이름"
                    }
                },
                "required": ["city"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "사용자가 요청한 종목의 주가 정보를 조회한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "stock_name": {
                        "type": "string",
                        "description": "주가를 조회할 종목명"
                    }
                },
                "required": ["stock_name"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "save_result",
            "description": "사용자의 질문과 답변을 Markdown 파일로 저장한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {
                        "type": "string",
                        "description": "저장할 파일명. 예: result.md"
                    },
                    "question": {
                        "type": "string",
                        "description": "사용자가 질문한 내용"
                    },
                    "answer": {
                        "type": "string",
                        "description": "저장할 답변 내용"
                    }
                },
                "required": ["filename", "question", "answer"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "read_result",
            "description": "저장된 Markdown 파일의 내용을 조회한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {
                        "type": "string",
                        "description": "조회할 파일명. 예: result.md"
                    }
                },
                "required": ["filename"],
                "additionalProperties": False
            }
        }
    }
]

In [47]:
available_tools = {
    "recommend_cloth": recommend_cloth,
    "check_data": check_data,
    "get_weather": get_weather,
    "get_stock_price": get_stock_price,
    "save_result": save_result,
    "read_result": read_result
}

def get_stock_price(stock_name):
    # 주가 조회 코드
    return f"{stock_name} 주가 조회 결과"


def save_result(filename, question, answer):
    with open(filename, "w", encoding="utf-8-sig") as file:
        file.write(f"# 질문\n\n{question}\n\n# 답변\n\n{answer}")

    return f"{filename}에 저장했습니다."


def read_result(filename):
    with open(filename, "r", encoding="utf-8-sig") as file:
        return file.read()

NameError: name 'get_stock_price' is not defined

In [51]:
# openai 내장 툴 사용
messages = [
    {'role': "user", "content": "부산 날씨 어때?"}
]

response = client.responses.create(
    model="gpt-5.6-luna",
    tools=[{"type" : "web_search"}],
    input=messages
)

In [52]:
print(response.output[2].content[0].text)

현재 부산은 **맑고 약 28°C**예요.  
오늘은 **최고 29°C, 최저 20°C**로 따뜻하고, 구름이 있다가 차차 맑아지겠습니다. 

내일은 흐리고 오후에 바람이 불며 **한때 소나기 가능성**이 있어요.


In [54]:
# openai 내장 툴 사용
messages = [
    {'role': "user", "content": "데이터 프레임에서 제목 알려줘"}
]

response = client.responses.create(
    model="gpt-5.6-luna",
    tools=[{"type" : "web_search"},
           {"type" :"function", 
            'name': 'check_data',
            'description': '데이터프레임에서 내용 조회할때 칼럼명으로 조회해서 알려준다. ',
            'parameters': {'type': 'object',
                'properties': {'column': {'type': 'string', 'description': '조회하고 싶은 칼럼'}},
                'required': ['column']}}
           ],
    input=messages
)

In [55]:
print(response)

Response(id='resp_06dfd94060a03ed2006aaa42c9179c87d09be85118245f24e8', created_at=1789543113.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.6-luna', object='response', output=[ResponseReasoningItem(id='rs_06dfd94060a03ed2006aaa42c9e2e887d0b0704634eaf09509', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABqqkLKcytMApymp5rfuuJxsPjbwptt6Zw9e20alnlDD6s47AELfYd3VPMTJcJCyxPTdK6x-tGG_-itBQbFtynKTz3taGENc9iun8Mmc679yns31sYGuTMCOF--noVOWCdVNyrza3Nke6m9kfI82uGxWtpUGnL19186_s-cYTwNmh-djs7pokZYgJDyXUiWtslRsJ5rTzeQJn6QocnsjsfGgG4YYncAzfT5AYcilFjZGsPyNlVJ64XI-jiM_ZqE9rK2nDxEjqkfucOy15Lvvm9VkdHFgu_3bRhNnhSTiBtYl2T0535SYVnnRRLmVrP8VHeWuWq-1hD37IZO6jC6Kirn7ZIHLYq_NVsmassKvBrtf6DduwDvnNn9lRzGn-f2SVBycZ1r0RwBL6wnke7q7pAGClhHhNdgN-nghUPmdUP0U9yL5_jnMWTAvIqKqr_4s0pRXDwqk2GX-Q8GeaBqg7CDHuGqrpzMiPw3SnJPNW51a8tRKS-we3bJE8dhXJFzA6teh6jblGLT0UiYEe9SkabTbgMDdVf3krQ_27lLuoRo3hn9xVqxf1wSkEDA2I5AT1YsGCxolvQCe5q4W-Mk8MIY6B7LFUz5O47IeBZvyJTJjJwL4hfDtAaenleYFnT

### 오픈 AI 제공 툴
- 내장 : web_search, code_interpreter 코드 실행 여부
- 직접 만든 함수 : 사내 사용 가능, 파일저장 및 수정 조회, 사내 DB 조회 -> 함수 만들어 사용
- MCP : 남이 공개한 도구

In [ ]:
# 없을 시 10까지 반복이라는 코드 작성 필요